## PROCESAMEINTO Y LIMPIEZA  DE FUENTES
-  Autor: Germán Homero Morán Figueroa
- Descripción: Este notebook permite realizar la limpieza y depuración  de la fuente Production_events.csv, con el objetivo de consolidar bases limpias para la contrucción de la vista minable que sirve de insumo para los modelo de Machine Learning
-  Salida: Dataframe eventos_cultivo.csv y dataframeCilima.csv, se almacenan en Oro.

In [24]:
# Librerias y dependencias
# =========================================================================
import pandas as pd
import numpy as np
from unidecode import  unidecode
import re
pd.options.display.max_columns = None

In [45]:
# Lectura de lso datsets
# ====================================================
Productions_Events = pd.read_csv("../../Data/Bronze/Production_events_4.csv")
fields = pd.read_csv("../../Data/Bronze/Fields_3.csv")
Productions_Events.tail(3)

,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA
995,4673,4321.0,4546.0,4785.0,8.853453,-75.751192,5/7/2016,Mecanizado,17.4,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,NaN,Algodón,NO,5/12/2016,59800.0,7/2/2016,9/23/2016,Mecanizada,6326.0,Grano seco,LA PALMERA,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,12652.0,17.0,NO,NaN,CÓRDOBA,CERETÉ,2.0
996,4674,4320.0,4545.0,4784.0,9.034286,-75.780458,5/9/2016,Mecanizado,18.5,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,DK 234 YGRR,6520.0,Algodón,SI,5/15/2016,70000.0,7/3/2016,9/21/2016,Mecanizada,6600.0,Grano seco,TIGRE,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14520.0,19.0,NO,NaN,CÓRDOBA,COTORRA,2.2
997,4675,4319.0,4544.0,4784.0,9.031350,-75.779953,5/9/2016,Mecanizado,17.8,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,P3966 (Pioneer),6400.0,Algodón,SI,5/14/2016,67600.0,7/5/2016,9/21/2016,Mecanizada,6440.0,Grano seco,CARACOL,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14812.0,18.0,NO,NaN,CÓRDOBA,COTORRA,2.3


In [46]:
print("Longitud del dataset lotes : ", fields.shape)
print("Longitud del datset eventos :",Productions_Events.shape)

Longitud del dataset lotes :  (4004, 10)
Longitud del datset eventos : (998, 44)


In [47]:
# Se Extraen las Columnas ID_LOTE y ALTURA_LOTE del datframe original
Altura_Lotes = fields[["ID_LOTE","ALTURA_LOT"]]
Altura_Lotes.head(3)

,ID_LOTE,ALTURA_LOT
0,1425,1638
1,1489,1637
2,861,1694


In [48]:
# Verificón de los productos cosechados. 
print(Productions_Events.PROD_COSECHADO.value_counts())

Grano seco          874
Ensilaje             59
Mazorca (fresca)     33
Name: PROD_COSECHADO, dtype: int64


### 1. Limpieza y trasnformaciónes Iniciales.

Inicialmente se realiza la limpieza y transformación de datos del archivo Production_events, que contienen toda la información detallada de los cultivos de maiz en diferentes lotes y granjas.

In [49]:
# Se obtienen una copia del dataframe original (Productions_Events) para su posterior procesamiento
eventos = Productions_Events.copy()


# Filtrar unicamente departamento de cordoba y municipios de interes. 
# ===========================================================================
''''
Se realiza limpieza de datos para las columnas departamento y municipio
Se eleiminan los acentos de las cadenas
'''
eventos.DEPARTAMENTO = eventos.DEPARTAMENTO.apply(lambda x: x.replace("Ó","O"))
eventos.MUNICIPIO = eventos.MUNICIPIO.apply(lambda s: unidecode(s))
# Lista con departamentos unicos
filDep = eventos.DEPARTAMENTO.unique()[0]
print(filDep)

'''
  Se filtra el dataset para que contenga los registros unicamente del departamento 
  de cordoba y los registros  asociados a producto [grano seco] se decartan los
  registros asociados a otros tipos de productos como [Ensilaje o Mazorca]

'''

evento_cordoba = eventos[(eventos.DEPARTAMENTO=='CORDOBA') & (eventos.PROD_COSECHADO=="Grano seco")]
print("Dimensiones: ",evento_cordoba.shape)
evento_cordoba.tail(5)

AttributeError: 'float' object has no attribute 'replace'

In [30]:
# Verificación cultivos anteriores en cada lote
print(evento_cordoba.CULT_ANT.unique())
print("Cantidad de Lotes Unicos: ", len(evento_cordoba.ID_LOTE.unique()))

['Algodón' 'Maiz' 'Frijol' 'Yuca' 'Pastos' nan]
Cantidad de Lotes Unicos:  885


In [31]:
# Se realiza el mismo tratamineto a la columna CULT_ANT
#evento_cordoba.CULT_ANT = evento_cordoba.CULT_ANT.astype(str)
#evento_cordoba.CULT_ANT = evento_cordoba.CULT_ANT.apply(lambda y: y.upper)

In [8]:
evento_cordoba[evento_cordoba.AREA < 1]

,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA
471,2939,2690,2565,2631,8.802194,-75.728350,5/4/2015,Manual,15.0,NO,0.80,0.2,Maiz,Blanco,2.0,NaN,NaN,Otro,6000.0,Maiz,NO,5/9/2015,60000.0,7/1/2015,9/24/2015,Manual,4500.0,Grano seco,MACHETICO,NaN,NaN,NaN,NaN,Hibrido,DK 234 VTPro,NaN,1.0,4500.0,20.0,NO,OBTUVO ESTOS RENDIMIENTOS DEBIDO A LA EPOCA CR...,CORDOBA,SAN CARLOS,0.5
473,2941,2692,2567,2633,8.821444,-75.751889,5/8/2015,Manual,16.0,SI,0.80,0.2,Maiz,Blanco,1.0,NaN,NaN,P3966 (Pioneer),7000.0,Maiz,SI,5/14/2015,60000.0,7/6/2015,9/28/2015,Manual,4000.0,Grano seco,LA VICTORIA,NaN,NaN,NaN,Fungicidas,Hibrido,NaN,NaN,1.0,4000.0,20.0,SI,NaN,CORDOBA,SAN CARLOS,0.9
477,2945,2685,2570,2636,8.790583,-75.764306,5/5/2015,Manual,16.0,NO,0.75,0.2,Maiz,Blanco,1.0,NaN,NaN,Otro,6500.0,Maiz,NO,5/11/2015,60000.0,7/2/2015,9/25/2015,Manual,4300.0,Grano seco,COROZAL1,NaN,NaN,NaN,NaN,Hibrido,DK 234 VTPro,NaN,1.0,4300.0,19.0,SI,NaN,CORDOBA,SAN CARLOS,0.9


In [32]:

'''
- Se Eliminan los los lotes que tienen una extensión menor a una area (1 HA)
- los valores de rendimiento, poblacion y de Población no coinciden
'''
lista_Observaciones_Eliminar = [2685,2690,2692]
evento_cordoba = evento_cordoba[evento_cordoba.ID_LOTE.isin(lista_Observaciones_Eliminar)== False]
print(len(evento_cordoba.columns))
print("Longitud Eventos Cordoba: ",evento_cordoba.shape)
evento_cordoba.tail(5)


44
Longitud Eventos Cordoba:  (882, 44)


,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA
993,4671,4323,4548,4787,9.018844,-75.758531,5/21/2016,Manual,17.5,NO,0.8,0.40,Maiz,Amarillo,3.0,NaN,NaN,Otro,5300.0,Maiz,SI,5/25/2016,64200.0,7/13/2016,10/4/2016,Manual,7150.0,Grano seco,SANTA MARTA,NaN,NaN,NaN,NaN,Hibrido,SV-1035,NaN,NaN,7150.0,17.0,NO,NaN,CORDOBA,SAN PELAYO,1.0
994,4672,4322,4547,4786,8.871503,-75.772411,5/14/2016,Mecanizado,19.2,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,6980.0,Maiz,SI,5/20/2016,60000.0,7/8/2016,9/30/2016,Mecanizada,6200.0,Grano seco,SALSIPUEDES,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,13020.0,17.0,NO,NaN,CORDOBA,CERETE,2.1
995,4673,4321,4546,4785,8.853453,-75.751192,5/7/2016,Mecanizado,17.4,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,NaN,Algodón,NO,5/12/2016,59800.0,7/2/2016,9/23/2016,Mecanizada,6326.0,Grano seco,LA PALMERA,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,12652.0,17.0,NO,NaN,CORDOBA,CERETE,2.0
996,4674,4320,4545,4784,9.034286,-75.780458,5/9/2016,Mecanizado,18.5,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,DK 234 YGRR,6520.0,Algodón,SI,5/15/2016,70000.0,7/3/2016,9/21/2016,Mecanizada,6600.0,Grano seco,TIGRE,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14520.0,19.0,NO,NaN,CORDOBA,COTORRA,2.2
997,4675,4319,4544,4784,9.031350,-75.779953,5/9/2016,Mecanizado,17.8,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,P3966 (Pioneer),6400.0,Algodón,SI,5/14/2016,67600.0,7/5/2016,9/21/2016,Mecanizada,6440.0,Grano seco,CARACOL,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14812.0,18.0,NO,NaN,CORDOBA,COTORRA,2.3


In [34]:
''' 
    Segun recoemndación el valor del rendimiento real (Disminuye 14%) su peso 
    en relación a la Humedad.
    Se contruye una nueva columna con el rendimento real por cultivo.

'''

#  Ajuste Valor Real del Rendimeinto.
# ==============================================================================
evento_cordoba["RDT_AJUSTADO"] = (100 -evento_cordoba.HUMEDAD)/(100-14)*evento_cordoba.RDT
print(len(evento_cordoba.columns))
evento_cordoba.tail(5)

45


,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA,RDT_AJUSTADO
993,4671,4323,4548,4787,9.018844,-75.758531,5/21/2016,Manual,17.5,NO,0.8,0.40,Maiz,Amarillo,3.0,NaN,NaN,Otro,5300.0,Maiz,SI,5/25/2016,64200.0,7/13/2016,10/4/2016,Manual,7150.0,Grano seco,SANTA MARTA,NaN,NaN,NaN,NaN,Hibrido,SV-1035,NaN,NaN,7150.0,17.0,NO,NaN,CORDOBA,SAN PELAYO,1.0,6900.581395
994,4672,4322,4547,4786,8.871503,-75.772411,5/14/2016,Mecanizado,19.2,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,6980.0,Maiz,SI,5/20/2016,60000.0,7/8/2016,9/30/2016,Mecanizada,6200.0,Grano seco,SALSIPUEDES,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,13020.0,17.0,NO,NaN,CORDOBA,CERETE,2.1,5983.720930
995,4673,4321,4546,4785,8.853453,-75.751192,5/7/2016,Mecanizado,17.4,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,NaN,Algodón,NO,5/12/2016,59800.0,7/2/2016,9/23/2016,Mecanizada,6326.0,Grano seco,LA PALMERA,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,12652.0,17.0,NO,NaN,CORDOBA,CERETE,2.0,6105.325581
996,4674,4320,4545,4784,9.034286,-75.780458,5/9/2016,Mecanizado,18.5,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,DK 234 YGRR,6520.0,Algodón,SI,5/15/2016,70000.0,7/3/2016,9/21/2016,Mecanizada,6600.0,Grano seco,TIGRE,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14520.0,19.0,NO,NaN,CORDOBA,COTORRA,2.2,6216.279070
997,4675,4319,4544,4784,9.031350,-75.779953,5/9/2016,Mecanizado,17.8,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,P3966 (Pioneer),6400.0,Algodón,SI,5/14/2016,67600.0,7/5/2016,9/21/2016,Mecanizada,6440.0,Grano seco,CARACOL,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14812.0,18.0,NO,NaN,CORDOBA,COTORRA,2.3,6140.465116


### 2. AJUSTE FORMATO FECHAS
Se realiza la tranformación a formato datetime de cada una de las variables fechas del dataframe.
Inicilamnete se identifica las columnas que empiezan con el nombre fecha
- FECHA_SIEMBRA
- FECHA_EMERGENCIA
- FECHA_FLORACION
- FECHA_COSECHA

In [35]:
# Verificación de Columnas
evento_cordoba.columns

Index(['ID_EVENTO', 'ID_LOTE', 'ID_FINCA', 'ID_PROD', 'LAT_LOTE', 'LONG_LOTE',
       'FECHA_SIEMBRA', 'TIPO_SIEMBRA', 'NUM_SEMILLAS', 'SEM_TRATADAS',
       'DIST_SURCOS', 'DIST_PLANTAS', 'TIPO_CULTIVO', 'COLOR_ENDOSPERMO',
       'SEM_POR_SITIO', 'TIPO_DE_SEMILLA', 'HABITO_CRECIMIENTO',
       'MATERIAL_GENETICO', 'OBJ_RDT', 'CULT_ANT', 'DRENAJE',
       'FECHA_EMERGENCIA', 'POBLACION_20DIAS', 'FECHA_FLORACION',
       'FECHA_COSECHA', 'METODO_COSECHA', 'RDT', 'PROD_COSECHADO',
       'NOMBRE_LOTE', 'ORIGEN_SEMILLA', 'INOCULACION_SEMILLAS',
       'NUEVA_INOCULACION_SEMILLAS', 'PRODUCTO_USADO', 'TIPO_MATERIAL',
       'NUEVO_MATERIAL_GENETICO', 'OTRO_CULT_ANT', 'RESIEMBRA',
       'CANTIDAD_TOTAL', 'HUMEDAD', 'ALMACENAMIENTO_FINCA',
       'OBSERVACIONES_COSECHA', 'DEPARTAMENTO', 'MUNICIPIO', 'AREA',
       'RDT_AJUSTADO'],
      dtype='object')

In [36]:
# Se verifica el tipo de dato de als columnas
evento_cordoba.FECHA_SIEMBRA

0      5/13/2013
1       5/2/2013
2      5/12/2013
3       5/7/2013
4       5/7/2013
         ...    
993    5/21/2016
994    5/14/2016
995     5/7/2016
996     5/9/2016
997     5/9/2016
Name: FECHA_SIEMBRA, Length: 882, dtype: object

In [37]:
# se realiza la conversion de las columnas fecha a tipo data-time
evento_cordoba.FECHA_SIEMBRA = pd.to_datetime(evento_cordoba.FECHA_SIEMBRA, infer_datetime_format=True)
evento_cordoba.FECHA_EMERGENCIA = pd.to_datetime(evento_cordoba.FECHA_EMERGENCIA, infer_datetime_format=True)
evento_cordoba.FECHA_FLORACION = pd.to_datetime(evento_cordoba.FECHA_FLORACION, infer_datetime_format=True)
evento_cordoba.FECHA_COSECHA = pd.to_datetime(evento_cordoba.FECHA_COSECHA, infer_datetime_format=True)


In [38]:
# Se calcula la diferencia en dias entre cada una de las etapas del cultivo

''' 
    Se definen diferentes etapas de cultivo  para cada una de las fechas
'''
evento_cordoba["DIAS_EN_EMERGER"] = (evento_cordoba.FECHA_EMERGENCIA- evento_cordoba.FECHA_SIEMBRA).dt.days
evento_cordoba["DIAS_EN_EMERGER_A_FLORECER"] = (evento_cordoba.FECHA_FLORACION- evento_cordoba.FECHA_EMERGENCIA).dt.days
evento_cordoba["DIAS_EN_FLORECER_A_COSECHAR"] = (evento_cordoba.FECHA_COSECHA- evento_cordoba.FECHA_FLORACION).dt.days

In [39]:
# Verificación cambios
evento_cordoba.tail(3)

,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR
995,4673,4321,4546,4785,8.853453,-75.751192,2016-05-07,Mecanizado,17.4,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,NaN,Algodón,NO,2016-05-12,59800.0,2016-07-02,2016-09-23,Mecanizada,6326.0,Grano seco,LA PALMERA,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,12652.0,17.0,NO,NaN,CORDOBA,CERETE,2.0,6105.325581,5.0,51.0,83.0
996,4674,4320,4545,4784,9.034286,-75.780458,2016-05-09,Mecanizado,18.5,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,DK 234 YGRR,6520.0,Algodón,SI,2016-05-15,70000.0,2016-07-03,2016-09-21,Mecanizada,6600.0,Grano seco,TIGRE,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14520.0,19.0,NO,NaN,CORDOBA,COTORRA,2.2,6216.279070,6.0,49.0,80.0
997,4675,4319,4544,4784,9.031350,-75.779953,2016-05-09,Mecanizado,17.8,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,P3966 (Pioneer),6400.0,Algodón,SI,2016-05-14,67600.0,2016-07-05,2016-09-21,Mecanizada,6440.0,Grano seco,CARACOL,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14812.0,18.0,NO,NaN,CORDOBA,COTORRA,2.3,6140.465116,5.0,52.0,78.0


In [40]:

''' 
    -Segun observaciones de los agricultores  de la FIAT Y FENALCE en una AREA [1HA] no puede existir mas de  90.000 plantas.
    Por tal motivo puede darse el caso que un agricultor no realizo adecuadamente la captura de estos datos e información
    se debe Ajustar ese valor  de la población de acuerdo al area.

    -Se crea una nueva columna con la población ajustada


'''

# Funcion para realizar el ajuste de la población a los 20 Dias.

def PoblacionAjustada(pob_20_dias,area):

    if pob_20_dias < 90000:
        poblacion = pob_20_dias
    else:
        poblacion = pob_20_dias/area
    
    return poblacion



# Se realiza el ajuste de la población a los 20 dias

evento_cordoba["POBLACION_20DIAS_AJT"] = evento_cordoba.POBLACION_20DIAS.apply(lambda x: PoblacionAjustada(x,evento_cordoba.AREA))



In [41]:
# Se recuperan unicamnete los registros pertenecientes al departamento de cordoba
# =================================================================================
print("Ids Unicos extraidos del fichero LOTES (Fields): ",len(Altura_Lotes.ID_LOTE.unique()))
print("IDs unicos de lotes del departamento de Cordoba: ", len(evento_cordoba.ID_LOTE.unique()))

# Lotes unicos departamento de Cordoba
# =====================================
Lista_ID_unicos_cordoba = list (evento_cordoba.ID_LOTE.unique())

# Se recupera la altura del lote  unicamente para lotes de cordoba
# ================================================================
Altura_Lotes = Altura_Lotes[Altura_Lotes.ID_LOTE.isin(Lista_ID_unicos_cordoba)]


Ids Unicos extraidos del fichero LOTES (Fields):  4004
IDs unicos de lotes del departamento de Cordoba:  882


In [42]:
# Adición Variable Altura del lote al dataframe eventos_cordoba
# =============================================================
evento_cordoba = pd.merge( evento_cordoba,Altura_Lotes)
print("Longitud nuevo archivo adicionado las nuevas columnas ",evento_cordoba.shape)
print("Cantidad de variables antes de eliminar: ",len(evento_cordoba.columns))
evento_cordoba

Longitud nuevo archivo adicionado las nuevas columnas  (882, 50)
Cantidad de variables antes de eliminar:  50


,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,LAT_LOTE,LONG_LOTE,FECHA_SIEMBRA,TIPO_SIEMBRA,NUM_SEMILLAS,SEM_TRATADAS,DIST_SURCOS,DIST_PLANTAS,TIPO_CULTIVO,COLOR_ENDOSPERMO,SEM_POR_SITIO,TIPO_DE_SEMILLA,HABITO_CRECIMIENTO,MATERIAL_GENETICO,OBJ_RDT,CULT_ANT,DRENAJE,FECHA_EMERGENCIA,POBLACION_20DIAS,FECHA_FLORACION,FECHA_COSECHA,METODO_COSECHA,RDT,PROD_COSECHADO,NOMBRE_LOTE,ORIGEN_SEMILLA,INOCULACION_SEMILLAS,NUEVA_INOCULACION_SEMILLAS,PRODUCTO_USADO,TIPO_MATERIAL,NUEVO_MATERIAL_GENETICO,OTRO_CULT_ANT,RESIEMBRA,CANTIDAD_TOTAL,HUMEDAD,ALMACENAMIENTO_FINCA,OBSERVACIONES_COSECHA,DEPARTAMENTO,MUNICIPIO,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT
0,53,40,42,13,8.877222,-75.764444,2013-05-13,Mecanizado,60000.0,NO,0.8,0.20,Maiz,Blanco,2.0,NaN,NaN,PIONEER 30F32,5000.0,Algodón,SI,2013-05-18,60000.0,2013-07-20,2013-09-26,Manual,5000.0,Grano seco,VILLA GABRIELA,NaN,NaN,NaN,NaN,Hibrido,NaN,NaN,NaN,5000.0,18.0,NO,NaN,CORDOBA,CERETE,1.0,4767.441860,5.0,63.0,68.0,60000.0,13
1,54,43,43,14,8.879167,-75.765556,2013-05-02,Mecanizado,60000.0,SI,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,DK 234,5.0,Maiz,SI,2013-05-07,60000.0,2013-07-10,2013-09-11,Manual,5000.0,Grano seco,VILLA LOURDES,NaN,NaN,NaN,Insecticidas,Hibrido,NaN,NaN,NaN,5000.0,20.0,NO,NaN,CORDOBA,CERETE,1.0,4651.162791,5.0,64.0,63.0,60000.0,15
2,56,44,44,15,8.880000,-75.765833,2013-05-12,Mecanizado,60000.0,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,PIONEER 30F32,5.0,Algodón,SI,2013-05-17,60000.0,2013-07-15,2013-09-19,Manual,5500.0,Grano seco,SANTA MARTA,NaN,NaN,NaN,NaN,Hibrido,NaN,NaN,NaN,5500.0,19.0,NO,NaN,CORDOBA,CERETE,1.0,5180.232558,5.0,59.0,66.0,60000.0,12
3,57,45,45,16,8.878611,-75.758889,2013-05-07,Mecanizado,60000.0,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,5.0,Algodón,SI,2013-05-12,60000.0,2013-07-15,2013-09-12,Manual,5200.0,Grano seco,PALMAR,NaN,NaN,NaN,NaN,Hibrido,7019,NaN,NaN,5200.0,19.0,NO,NaN,CORDOBA,CERETE,1.0,4897.674419,5.0,64.0,59.0,60000.0,12
4,273,46,46,17,8.876389,-75.759167,2013-05-07,Mecanizado,60000.0,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,5000.0,Algodón,SI,2013-05-12,60000.0,2013-07-14,2013-09-12,Manual,5700.0,Grano seco,DONDE FIDEL,NaN,NaN,NaN,NaN,Hibrido,7019,NaN,NaN,5700.0,20.0,NO,NaN,CORDOBA,CERETE,1.0,5302.325581,5.0,63.0,60.0,60000.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,4671,4323,4548,4787,9.018844,-75.758531,2016-05-21,Manual,17.5,NO,0.8,0.40,Maiz,Amarillo,3.0,NaN,NaN,Otro,5300.0,Maiz,SI,2016-05-25,64200.0,2016-07-13,2016-10-04,Manual,7150.0,Grano seco,SANTA MARTA,NaN,NaN,NaN,NaN,Hibrido,SV-1035,NaN,NaN,7150.0,17.0,NO,NaN,CORDOBA,SAN PELAYO,1.0,6900.581395,4.0,49.0,83.0,64200.0,18
878,4672,4322,4547,4786,8.871503,-75.772411,2016-05-14,Mecanizado,19.2,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,6980.0,Maiz,SI,2016-05-20,60000.0,2016-07-08,2016-09-30,Mecanizada,6200.0,Grano seco,SALSIPUEDES,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,13020.0,17.0,NO,NaN,CORDOBA,CERETE,2.1,5983.720930,6.0,49.0,84.0,60000.0,16
879,4673,4321,4546,4785,8.853453,-75.751192,2016-05-07,Mecanizado,17.4,NO,0.8,0.20,Maiz,Blanco,1.0,NaN,NaN,Otro,NaN,Algodón,NO,2016-05-12,59800.0,2016-07-02,2016-09-23,Mecanizada,6326.0,Grano seco,LA PALMERA,NaN,NaN,NaN,NaN,Hibrido,SV-7019,NaN,NaN,12652.0,17.0,NO,NaN,CORDOBA,CERETE,2.0,6105.325581,5.0,51.0,83.0,59800.0,16
880,4674,4320,4545,4784,9.034286,-75.780458,2016-05-09,Mecanizado,18.5,NO,0.8,0.18,Maiz,Blanco,1.0,NaN,NaN,DK 234 YGRR,6520.0,Algodón,SI,2016-05-15,70000.0,2016-07-03,2016-09-21,Mecanizada,6600.0,Grano seco,TIGRE,NaN,NaN,NaN,NaN,OGM,NaN,NaN,NaN,14520.0,19.0,NO,NaN,CORDOBA,COTORRA,2.2,6216.279070,6.0,49.0,80.0,70000.0,17


In [43]:

''' 
 Se construyen 2 nuevos dataframes que seran utilizados para cruzes posteriores
 - eventosParaFechas: Contiene las fechas principales asociadas a cada una de las etapas del cultivo
 - Variables Clima: Contiene columnas que seran utilizadas para realizar el calculos climaticos.

'''

# Se extraen las columnas clave , que se seran utilizadas para cruzar otro tipo de información | Solo se extraen los eventos para las fechas
eventosParaFechas =evento_cordoba[["ID_EVENTO","ID_LOTE","FECHA_SIEMBRA","FECHA_EMERGENCIA","FECHA_COSECHA","FECHA_FLORACION"]]
print("Longitud eventosParafechas", eventosParaFechas.shape)
eventosParaFechas.tail(2)

Longitud eventosParafechas (882, 6)


,ID_EVENTO,ID_LOTE,FECHA_SIEMBRA,FECHA_EMERGENCIA,FECHA_COSECHA,FECHA_FLORACION
880,4674,4320,2016-05-09,2016-05-15,2016-09-21,2016-07-03
881,4675,4319,2016-05-09,2016-05-14,2016-09-21,2016-07-05


In [24]:
# Se seleccionan las columnas que podran ser utilizadas para  calculos climaticos 
variablesClima = ["ID_EVENTO","ID_LOTE","ID_FINCA","ID_PROD","FECHA_SIEMBRA","LAT_LOTE","LONG_LOTE","FECHA_EMERGENCIA","FECHA_FLORACION","FECHA_COSECHA","DEPARTAMENTO","MUNICIPIO","PROD_COSECHADO","RDT_AJUSTADO"]
evenParaClima = evento_cordoba[variablesClima]
print("Longitud eventosParaClima",evenParaClima.shape)
evenParaClima.tail(2)


Longitud eventosParaClima (882, 14)


,ID_EVENTO,ID_LOTE,ID_FINCA,ID_PROD,FECHA_SIEMBRA,LAT_LOTE,LONG_LOTE,FECHA_EMERGENCIA,FECHA_FLORACION,FECHA_COSECHA,DEPARTAMENTO,MUNICIPIO,PROD_COSECHADO,RDT_AJUSTADO
880,4674,4320,4545,4784,2016-05-09,9.034286,-75.780458,2016-05-15,2016-07-03,2016-09-21,CORDOBA,COTORRA,Grano seco,6216.279070
881,4675,4319,4544,4784,2016-05-09,9.031350,-75.779953,2016-05-14,2016-07-05,2016-09-21,CORDOBA,COTORRA,Grano seco,6140.465116


### COLUMNAS A ELIMINAR
Se tienen en cuenta las siguientes consideraciones:

- Se eliminan las siguientes columnas por ser identificadores: ID_PROD, ID_FINCA, DEPARTAMENTO, MUNICIPIO
- Por ser utilizadas para la generacion de columnas derivadas: LAT_LOTE, LONG_LOTE, RDT, FECHA_SIEMBRA, FECHA_EMERGENCIA, FECHA_FLORACION, FECHA_COSECHA, POBLACION_20DIAS, HUMEDAD, RDT, CANTIDAD_TOTAL
- Gran cantidad de datos faltantes: RESIEMBRA, OBSERVACIONES_COSECHA
- Inconsistencias en las unidades de medidas: NUM_SEMILLAS v2
- No aportan información relevante, ademas  las undiades y medidas generan isnconsia, la información recolectada no es fiable segun expertos de la FIAT y FENALCE: DIST_PLANTAS,DIST_SURCOS,SEM_POR_SITIO,COLOR_ENDOSPERMO,TIPO_MATERIAL

In [44]:
# Se eliminan las columnas del dataframe eventos_cordoba
# =======================================================
VariablesEliminar  = ["ID_PROD","ID_FINCA","LAT_LOTE","LONG_LOTE","TIPO_CULTIVO","TIPO_DE_SEMILLA","HABITO_CRECIMIENTO","OBJ_RDT","RDT","NOMBRE_LOTE","ORIGEN_SEMILLA","INOCULACION_SEMILLAS","NUEVA_INOCULACION_SEMILLAS","HUMEDAD","OBSERVACIONES_COSECHA","DEPARTAMENTO","MUNICIPIO",
                     "NUM_SEMILLAS","CANTIDAD_TOTAL","FECHA_SIEMBRA","FECHA_EMERGENCIA","FECHA_FLORACION","FECHA_COSECHA","PROD_COSECHADO","NUEVO_MATERIAL_GENETICO","OTRO_CULT_ANT","PRODUCTO_USADO","RESIEMBRA","POBLACION_20DIAS","DIST_PLANTAS","DIST_SURCOS"
                     ,"SEM_POR_SITIO","COLOR_ENDOSPERMO","TIPO_MATERIAL"]

print("Variables a Eliminar: ",len(VariablesEliminar))
evento_cordoba = evento_cordoba.drop(VariablesEliminar,axis=1)
print("Cantidad de variables despues de eliminar: ",len(evento_cordoba.columns))
evento_cordoba.head(5)

Variables a Eliminar:  34
Cantidad de variables despues de eliminar:  16


,ID_EVENTO,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT
0,53,40,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,4767.441860,5.0,63.0,68.0,60000.0,13
1,54,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,1.0,4651.162791,5.0,64.0,63.0,60000.0,15
2,56,44,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,5180.232558,5.0,59.0,66.0,60000.0,12
3,57,45,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,4897.674419,5.0,64.0,59.0,60000.0,12
4,273,46,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,5302.325581,5.0,63.0,60.0,60000.0,16


In [26]:
# Guardamos los dataframe de clima y eventos_cordoba para posterior analisis y procesamiento
# ===========================================================================================
evenParaClima.to_csv("../../Data/Silver/cordoba_clima.csv", index=False)
evento_cordoba.to_csv("../../Data/Silver/eventos_cordoba_2017.csv", index=False)
eventosParaFechas.to_csv("../../Data/Silver/cordoba_fechas_stages.csv", index=False)